# Optimizers & LR schedules: hands-on (optional)
**LLMs From Basics · Lesson M2.3**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (plus a ✓) means success.

Stuck? Solutions are at the very bottom. Try first, though: *struggling to recall is what makes it stick.*

Only `numpy` and `matplotlib` are needed (both pre-installed on Colab).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print("ready")

## 1 · SGD and momentum
Every optimizer here takes the current parameters `theta` (a NumPy array) and a gradient `g`, and returns new parameters (plus any state it keeps).

$$\text{SGD:}\quad \theta \leftarrow \theta - \eta\,g \qquad\qquad \text{Momentum:}\quad v \leftarrow \mu v + g,\quad \theta \leftarrow \theta - \eta\,v$$

In [ ]:
def sgd_step(theta, g, lr):
    # TODO: return the new theta
    raise NotImplementedError("Exercise 1a")

def momentum_step(theta, v, g, lr, mu=0.9):
    # TODO: update the velocity v, then theta; return (theta, v)
    raise NotImplementedError("Exercise 1b")

**Check 1**

In [ ]:
t = np.array([1.0, -2.0])
assert np.allclose(sgd_step(t, np.array([0.5, 1.0]), 0.1), [0.95, -2.1])
th, v = np.array([0.0]), np.array([0.0])
for g in [1.0, 1.0, 1.0]:                       # a constant gradient: velocity builds up
    th, v = momentum_step(th, v, np.array([g]), lr=0.1, mu=0.9)
assert np.allclose(v, [1 + 0.9 + 0.81]) and np.allclose(th, [-0.1 * (1 + 1.9 + 2.71)])
print("✓ exercise 1")

## 2 · Adam, with bias correction
$$m \leftarrow \beta_1 m + (1-\beta_1) g, \qquad v \leftarrow \beta_2 v + (1-\beta_2) g^2$$
$$\hat m = \frac{m}{1-\beta_1^t}, \qquad \hat v = \frac{v}{1-\beta_2^t}, \qquad \theta \leftarrow \theta - \eta\,\frac{\hat m}{\sqrt{\hat v}+\epsilon}$$
`t` counts steps starting at **1**. Everything is element-wise.

In [ ]:
def adam_step(theta, g, m, v, t, lr, b1=0.9, b2=0.95, eps=1e-8):
    # TODO: update m and v, bias-correct them, update theta; return (theta, m, v)
    raise NotImplementedError("Exercise 2")

**Check 2.** These are the lesson's worked and faded examples ($g_1 = 2$, then $g_2 = 1$, $\eta = 0.01$).

In [ ]:
th, m, v = np.array([0.0]), np.zeros(1), np.zeros(1)
th, m, v = adam_step(th, np.array([2.0]), m, v, t=1, lr=0.01)
assert np.allclose(th, [-0.01]), "the first Adam step should be exactly lr"
th2, m, v = adam_step(th, np.array([1.0]), m, v, t=2, lr=0.01)
assert np.allclose(m, [0.28]) and np.allclose(v, [0.24])
assert abs((th - th2)[0] - 0.009393) < 1e-5
# the first step has size lr whatever the gradient's scale
for g in [1e-4, 3.0, 1e4]:
    s, _, _ = adam_step(np.zeros(1), np.array([g]), np.zeros(1), np.zeros(1), t=1, lr=0.01)
    assert np.allclose(s, [-0.01], rtol=1e-3)
print("✓ exercise 2")

## 3 · AdamW: decoupled weight decay
AdamW adds the decay **directly** to the update instead of to the gradient:
$$\theta \leftarrow \theta - \eta\left(\frac{\hat m}{\sqrt{\hat v}+\epsilon} + \lambda\,\theta\right)$$
Implement it by reusing your Adam maths. Then compare with the old way, L2 regularization, which adds $\lambda\theta$ to the gradient (provided below).

In [ ]:
def adamw_step(theta, g, m, v, t, lr, wd=0.1, b1=0.9, b2=0.95, eps=1e-8):
    # TODO: Adam update on g (no decay in g!), plus the decoupled term lr * wd * theta
    #       (use the theta from *before* this step). Return (theta, m, v).
    raise NotImplementedError("Exercise 3")

In [ ]:
def adam_l2_step(theta, g, m, v, t, lr, wd=0.1, **kw):
    return adam_step(theta, g + wd * theta, m, v, t, lr, **kw)   # decay goes through the gradient

**Check 3.** With a zero gradient, AdamW shrinks a weight by exactly a factor $(1-\eta\lambda)$ per step. Adam + L2 instead normalizes the decay term, so it moves by about $\eta$ per step whatever the weight's size: a big weight is barely regularized.

In [ ]:
w, m, v = np.array([5.0]), np.zeros(1), np.zeros(1)
w1, _, _ = adamw_step(w, np.zeros(1), m, v, t=1, lr=0.01, wd=0.1)
assert np.allclose(w1, 5.0 * (1 - 0.01 * 0.1))
w2, _, _ = adam_l2_step(w, np.zeros(1), m, v, t=1, lr=0.01, wd=0.1)
assert np.allclose(w2, 5.0 - 0.01, atol=1e-6), "Adam+L2 moves by ~lr, regardless of the weight"
print("✓ exercise 3 · AdamW decay:", round(float((w - w1)[0]), 6), "  Adam+L2 decay:", round(float((w - w2)[0]), 6))

## 4 · Race on badly scaled gradients
Loss $L(\theta) = \tfrac12 \sum_i s_i \theta_i^2$ with curvatures $s = (100, 1, 0.01)$: gradients differ by a factor of 10,000 between coordinates, a miniature of what happens across an LLM's parameters. SGD's learning rate must stay below $2/100 = 0.02$ or the first coordinate diverges, so the last one barely moves.

Fill in the training loop: for each step compute the gradient $s \odot \theta$, call the optimizer, and record the loss.

In [ ]:
S = np.array([100.0, 1.0, 0.01])
def loss_fn(th): return 0.5 * np.sum(S * th ** 2)
def grad_fn(th): return S * th

def make_sgd(lr):
    return lambda th, g, state, t: (sgd_step(th, g, lr), state)
def make_momentum(lr, mu=0.9):
    return lambda th, g, state, t: momentum_step(th, state, g, lr, mu)
def make_adam(lr):
    def step(th, g, state, t):
        m, v = state if state is not None else (np.zeros_like(th), np.zeros_like(th))
        th, m, v = adam_step(th, g, m, v, t, lr)
        return th, (m, v)
    return step

In [ ]:
def train(step_fn, steps=500, init_state=None):
    th, state, losses = np.ones(3), init_state, [loss_fn(np.ones(3))]
    for t in range(1, steps + 1):
        # TODO: gradient -> optimizer step (pass t, starting at 1) -> append loss
        raise NotImplementedError("Exercise 4")
    return th, losses

**Check 4**

In [ ]:
runs = {"SGD lr=0.019": train(make_sgd(0.019)),
        "momentum lr=0.002": train(make_momentum(0.002), init_state=np.zeros(3)),
        "Adam lr=0.01": train(make_adam(0.01))}
for name, (th, losses) in runs.items():
    plt.semilogy(losses, label=name); print(f"{name:18s} final θ = {np.round(th, 4)}")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("Badly scaled problem"); plt.show()
assert abs(runs["SGD lr=0.019"][0][2]) > 0.85, "SGD should barely move the flat coordinate"
assert abs(runs["Adam lr=0.01"][0][2]) < 0.1, "Adam should move every coordinate at about the same speed"
assert train(make_sgd(0.021), steps=300)[1][-1] > 1e3, "lr above 2/100 should diverge"
print("✓ exercise 4")

## 5 · Warmup + cosine schedule, and gradient clipping
For step $t$ of $T$, with $T_w$ warmup steps:
$$\eta_t = \eta_{\max}\,\frac{t}{T_w}\ \ (t < T_w), \qquad \eta_t = \eta_{\min} + \tfrac12(\eta_{\max}-\eta_{\min})\left(1+\cos\frac{\pi (t-T_w)}{T-T_w}\right)\ \ (t \ge T_w)$$
Clipping by global norm: $g \leftarrow g \cdot \min(1, c/\lVert g\rVert)$, with $\lVert g\rVert$ taken over **all** parameter arrays together.

In [ ]:
def lr_at(t, total, warmup, peak, final):
    # TODO: linear warmup, then cosine decay from peak to final
    raise NotImplementedError("Exercise 5a")

def clip_by_global_norm(grads, c=1.0):
    # grads is a list of arrays. TODO: return (clipped list, original global norm)
    raise NotImplementedError("Exercise 5b")

**Check 5**

In [ ]:
T, TW, PEAK, FINAL = 100_000, 2_000, 3e-4, 3e-5
assert lr_at(0, T, TW, PEAK, FINAL) == 0
assert np.isclose(lr_at(1_000, T, TW, PEAK, FINAL), 1.5e-4)
assert np.isclose(lr_at(TW, T, TW, PEAK, FINAL), PEAK)
assert np.isclose(lr_at(T, T, TW, PEAK, FINAL), FINAL)
assert np.isclose(lr_at(TW + (T - TW) // 2, T, TW, PEAK, FINAL), (PEAK + FINAL) / 2)
lrs = [lr_at(t, T, TW, PEAK, FINAL) for t in range(T + 1)]
assert all(a >= b for a, b in zip(lrs[TW:], lrs[TW + 1:])), "should only decrease after warmup"
(c1, c2), n = clip_by_global_norm([np.array([3.0]), np.array([4.0])], c=1.0)
assert np.isclose(n, 5) and np.allclose([c1[0], c2[0]], [0.6, 0.8])
(s,), n = clip_by_global_norm([np.array([0.3, 0.4])], c=1.0)
assert np.allclose(s, [0.3, 0.4]), "small gradients pass through unchanged"
plt.plot(lrs); plt.xlabel("step"); plt.ylabel("learning rate"); plt.title("Warmup + cosine"); plt.show()
print("✓ exercise 5")

## Experiment
Try these and watch what changes:
- In exercise 4, raise SGD's learning rate to 0.021. Which coordinate blows up, and why that one?
- Give Adam `b2=0.999` (the paper's default) instead of 0.95 by editing `make_adam`. Does anything change on this smooth problem? (The difference matters most with noisy gradients and sudden gradient-scale changes, as in large LLM runs.)
- Plot `lr_at` for warmup 0, 2,000 and 20,000 steps, and for `final` = 0 vs 10% of peak.

In [ ]:
# Your experiments here
for tw in [1, 2_000, 20_000]:
    plt.plot([lr_at(t, T, tw, PEAK, FINAL) for t in range(0, T + 1, 100)], label=f"warmup {tw}")
plt.legend(); plt.xlabel("step (×100)"); plt.ylabel("learning rate"); plt.show()

## Recap
- **Momentum** averages gradients: oscillations cancel, consistent directions build up.
- **Adam** divides by the running RMS gradient, so every coordinate moves about `lr` per step; **bias correction** fixes the zero start.
- **AdamW** shrinks weights by a fixed fraction per step, independent of gradient history.
- Real runs use **warmup + decay** and **clip** the global gradient norm (usually at 1.0).

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def sgd_step(theta, g, lr):
    return theta - lr * g

def momentum_step(theta, v, g, lr, mu=0.9):
    v = mu * v + g
    return theta - lr * v, v

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def adam_step(theta, g, m, v, t, lr, b1=0.9, b2=0.95, eps=1e-8):
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * g * g
    m_hat = m / (1 - b1 ** t)
    v_hat = v / (1 - b2 ** t)
    return theta - lr * m_hat / (np.sqrt(v_hat) + eps), m, v

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def adamw_step(theta, g, m, v, t, lr, wd=0.1, b1=0.9, b2=0.95, eps=1e-8):
    new, m, v = adam_step(theta, g, m, v, t, lr, b1, b2, eps)
    return new - lr * wd * theta, m, v

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def train(step_fn, steps=500, init_state=None):
    th, state, losses = np.ones(3), init_state, [loss_fn(np.ones(3))]
    for t in range(1, steps + 1):
        g = grad_fn(th)
        th, state = step_fn(th, g, state, t)
        losses.append(loss_fn(th))
    return th, losses

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def lr_at(t, total, warmup, peak, final):
    if t < warmup:
        return peak * t / warmup
    q = (t - warmup) / (total - warmup)
    return final + 0.5 * (peak - final) * (1 + np.cos(np.pi * q))

def clip_by_global_norm(grads, c=1.0):
    norm = np.sqrt(sum(np.sum(g ** 2) for g in grads))
    scale = min(1.0, c / (norm + 1e-12))
    return [g * scale for g in grads], norm